# 07 — Reflexão de treino aplicada à validação (AQuA, pareamento por similaridade)

**Insumo:** `data/aqua_df_most_similar_validation_train_questions_with_answers.csv` — para cada
uma das 254 questões de **validação** do AQuA, a questão de **treino** (de um pool bem maior que
os 383 itens oficiais do projeto — ver célula de proveniência abaixo) mais similar por embedding,
já com o gabarito (`validation_correct` / `train_correct`) de cada lado. `rank` é sempre 1: este
CSV já decidiu a recuperação (top-1), não há `k` para variar aqui.

**Pipeline pedido pelo usuário, em 4 etapas:**

1. `llama3-8b` responde as questões de **treino** (254 linhas, 252 únicas).
2. Um modelo de linguagem **julga** se a resposta bateu com o gabarito (etapa separada da
   extração por regex — ver reasoning abaixo).
3. `llama3-8b` escreve uma **autorreflexão** sobre cada resposta de treino (certa ou errada).
4. `llama3-8b` responde as 254 questões de **validação**, injetando a reflexão do seu par de
   treino mais similar — e cai no **prompt baseline puro** quando nenhuma reflexão é usada.
   O mesmo juiz da etapa 2 avalia essas respostas.

**Diferença deliberada em relação a `rmcq.common`:** o prompt de resposta aqui NÃO é
`ANSWER_PROMPT` (`FINAL ANSWER: <letra>`). É a instrução literal pedida nesta sessão —
`Answer: X` — adaptada para os 5 rótulos do AQuA (A–E, não A–D; ver célula de prompts). O
extrator (`rmcq.common.extract_final_answer`) já cobre esse formato via o método `answer_is`
sem nenhuma mudança de código — é só usado aqui para diagnóstico de aderência de formato, nunca
para decidir acerto (isso é o juiz, por pedido explícito do usuário).

**O que este notebook NÃO é:** não é o "07 de reflexão externa (gpt-5-petrobras)" que o resultado
da grade v3 apontou como o próximo eixo livre para ARC/LogiQA2 (ver memória do projeto,
`resultado_v3` / `reflexoes-auditoria-v1-v2`) — este é um experimento novo, num dataset novo
(AQuA, matemática/álgebra, nunca testado nas rodadas anteriores), com autorreflexão
(`llama3-8b` como aluno E professor de si mesmo), e retrieval JÁ FIXADO pelo CSV (top-1, sem
threshold de recuperação pré-computado). Por isso ganhou um nome próprio em vez de continuar a
numeração dos notebooks de reflexão externa.


## Reasoning: faz sentido um grid aqui?

**Resposta curta: um grid pequeno (profundidade × limiar × pool), sim — mas como checagem de
robustez em cima de UMA condição-âncora, não como garimpo da melhor célula.** A razão de cada
eixo:

**`k` (quantas reflexões recuperar) — FORA do grid, e não por escolha, por dado.** O CSV já fixa
`rank == 1` para as 254 linhas: a recuperação (top-1 por similaridade) já foi decidida antes de
este notebook existir. Rodar `k=3` exigiria recomputar embeddings e vizinhos, o que não foi
pedido e não está nos dados fornecidos. Isso por si só já mata metade dos eixos que a grade v3
(`scripts/run_grid_v3.py`) testou em ARC/LogiQA2.

**`threshold` de similaridade — ENTRA, e quase de graça.** O usuário pediu explicitamente:
"em caso em que nenhuma reflexão seja recuperada, o prompt deve ser o baseline". Como o retrieval
aqui é sempre top-1 (nunca vazio), a única forma de "nenhuma reflexão recuperada" acontecer é um
limiar mínimo de similaridade — abaixo dele, tratamos como não recuperado. A distribuição real de
similaridade no CSV (mínimo 0.50, Q1 0.72, mediana 0.81, Q3 0.91, máximo 1.00) dá 3 limiares
não-arbitrários para testar além do "sempre usar" (`0.0`): `0.70`, `0.80`, `0.90`.
O truque que torna isso barato: com decodificação gulosa (`temperature=0`, greedy — é o que
`STUDENT_GEN` já usa no projeto), o mesmo prompt sempre produz a mesma resposta. Então geramos
UMA vez a resposta "com reflexão" (incondicional, todas as 254 validações) e UMA vez a resposta
baseline; qualquer limiar é só decidir, por item, qual das duas reaproveitar — sem gastar GPU de
novo. É a mesma ideia de "uma geração por prompt distinto" que `scripts/run_grid_v3.py` usa
(ver memória do projeto, `grade-v3-script`), aplicada aqui manualmente porque a escala (254 itens)
não justifica reescrever o script completo.

**`pool` (injetar reflexão de toda resposta de treino vs. só das erradas) — ENTRA, também de
graça pelo mesmo motivo (é um filtro pós-hoc, não uma geração nova).** Vale incluir porque é o
fator mais replicado do histórico do projeto: em TRÊS rodadas (v1, v2, grade v3) notas sobre
respostas CORRETAS do treino se mostraram as mais nocivas, e `pool="errors"` bateu `pool="all"`
de forma consistente (embora nunca tenha ficado positivo). Testar se isso se repete num domínio
totalmente diferente (matemática/álgebra, AQuA) de um dado novo é uma pergunta que vale a pena, e
não custa nada gerar.

**`depth` (simple/complex) — ENTRA, mas por barateza, não porque se espera que importe.** As três
rodadas anteriores (v1, v2, grade v3) não conseguiram distinguir `simple` de `complex` em nenhum
teste. Incluir aqui é só uma checagem de robustez de ~252 gerações extras de reflexão — desprezível
perto do resto — não uma aposta de que vai mudar o resultado.

**O que isso dá: 2 × 4 × 2 = 16 células de grid, pelo custo computacional de ~4 condições
(baseline treino, baseline validação, reflexão×2 profundidades, validação-com-reflexão×2
profundidades).** Rodar como grid é essencially grátis; o CUSTO real está em como ele é
**interpretado**:

- **N pequeno.** 254 itens de validação é menor que os 298/300 do ARC/LogiQA2 nas rodadas
  anteriores, e `pool="errors"` reduz ainda mais (só os itens cujo par de treino o modelo errou).
  Com N pequeno, um p<0.05 isolado numa célula do grid não é evidência forte — é exatamente o
  padrão que a rodada v3 documentou (13/320 células com p<0.05, MENOS que o esperado por acaso).
- **16 testes = ~1 falso positivo esperado a α=0.05, só por acaso.** Não escolher a célula
  "vencedora" depois de olhar os números (ver `resultado_v3` na memória do projeto — essa
  disciplina já rendeu um resultado publicável nas rodadas anteriores, vale manter).
- **Por isso a célula 28 define uma "condição-âncora" ANTES de rodar** — `threshold=0.0` (usa a
  reflexão em toda validação, sem filtro) × `pool="all"`, por profundidade — como a comparação
  primária, pareada, contra o baseline. O resto do grid (as outras 14 células) é reportado como
  **análise de sensibilidade**, não como uma segunda chance de achar significância.

**Resumo prático:** grid de 16 células, sim — porque threshold e pool praticamente não custam
nada dado o reaproveitamento de geração gulosa, e cobrem exatamente os dois pedidos explícitos do
usuário (fallback pro baseline; e o fator mais replicado do histórico do projeto). Mas o notebook
trata só 2 células (`threshold=0.0, pool=all`, para os dois `depth`) como o resultado que decide
alguma coisa; as outras 14 são contexto.


## Setup

In [ ]:
import json
import math
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "rmcq").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "rmcq").exists(), f"rode este notebook a partir da raiz do repo (achei {REPO_ROOT})"
sys.path.insert(0, str(REPO_ROOT))

from rmcq.backends import GenParams, get_backend
from rmcq.common import (
    FEEDBACK_CORRECT,
    FEEDBACK_INCORRECT,
    REFLECTION_PROMPTS,
    compact_reflection,
    extract_final_answer,
    format_options,
    neutralize_option_letters,
    strip_think,
)
from rmcq.config import SEED, STUDENT_GEN, TEACHER_GEN
from rmcq.stages.analyze import utility
from rmcq.store import JsonlStore, get_logger, progress

log = get_logger("rmcq.notebook07")

# SMOKE_TEST=True troca o backend real pelo StubBackend (sem GPU, sem pesos) e limita os
# itens processados — serve para validar a canalização inteira em segundos, mesmo padrão dos
# notebooks 03/04/05. Trocar para False só quando for rodar de verdade (leva horas via vLLM).
SMOKE_TEST = True
SMOKE_LIMIT = 12  # itens de validação processados no modo smoke (treino segue o rank=1 deles)


## Config

In [ ]:
# --- Modelos -----------------------------------------------------------------
STUDENT_MODEL = "llama3-8b"  # quem responde e quem reflete sobre si mesmo (autorreflexão)
JUDGE_MODEL = "llama3-8b"    # quem julga acerto. Ver nota abaixo antes de rodar de verdade.

# Nota sobre o juiz: usar o MESMO modelo para responder e julgar é o default aqui só porque não
# exige nenhuma dependência nova (funciona com o StubBackend, sem GPU extra). Julgar a PRÓPRIA
# resposta é menos arriscado que auto-avaliar qualidade de texto (é um pareamento resposta<->
# gabarito, não uma opinião), mas ainda assim um juiz independente é mais confiável. Se este
# ambiente tiver acesso ao gateway Azure da Petrobras (ver memória `azure-petrobras`), troque
# JUDGE_MODEL para um deployment registrado em rmcq.config.MODELS (ex.: "gpt-4o-petrobras") e
# BACKEND_KIND_JUDGE para "azure" — nenhuma outra célula precisa mudar.
BACKEND_KIND_STUDENT = "stub" if SMOKE_TEST else None  # None = usa RMCQ_BACKEND do .env (vllm/hf)
BACKEND_KIND_JUDGE = "stub" if SMOKE_TEST else None

# --- Grid (ver célula de reasoning acima) -------------------------------------
DEPTHS = ("simple", "complex")
# Quartis reais da similaridade top-1 no CSV (ver célula de proveniência): 0.0 = sempre usa a
# reflexão; os outros três são Q1/mediana/Q3 arredondados, não chutados.
THRESHOLDS = (0.0, 0.70, 0.80, 0.90)
POOLS = ("all", "errors")  # "errors" = só injeta reflexão de treino cujo o modelo errou
ANCHOR_THRESHOLD = 0.0
ANCHOR_POOL = "all"

# --- Caminhos ------------------------------------------------------------------
DATA_CSV = REPO_ROOT / "data" / "aqua_df_most_similar_validation_train_questions_with_answers.csv"
OUT_ROOT = REPO_ROOT / "results" / "aqua_external"
BASELINE_DIR = OUT_ROOT / "baseline"
JUDGE_DIR = OUT_ROOT / "judge"
REFLECTIONS_DIR = OUT_ROOT / "reflections"
EVAL_DIR = OUT_ROOT / "eval_unconditional"
GRID_DIR = OUT_ROOT / "grid"
DIAG_DIR = OUT_ROOT / "diagnostics"
for d in (BASELINE_DIR, JUDGE_DIR, REFLECTIONS_DIR, EVAL_DIR, GRID_DIR, DIAG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"SMOKE_TEST={SMOKE_TEST}  STUDENT_MODEL={STUDENT_MODEL}  JUDGE_MODEL={JUDGE_MODEL}")
print(f"CSV: {DATA_CSV}  (existe: {DATA_CSV.exists()})")


## Carregar e interpretar o CSV

Cada linha do CSV traz o texto BRUTO da questão com as opções coladas (`"...\nA)32400\nB)6000\n..."`).
Precisamos separar enunciado de alternativas, e como o `train_index`/`validation_index` do CSV
**não** batem com a posição nos splits oficiais do projeto (`data/splits/aqua/*.jsonl` — checado
abaixo, é esperado: o pool de treino aqui é bem maior que os 383 itens oficiais), este notebook
constrói os itens diretamente do texto do CSV, sem tentar casar por índice.


In [ ]:
_OPT_RE = re.compile(r"(?:^|\n)([A-E])\)")


def parse_choices(raw_text: str) -> tuple[str, list[dict]]:
    """Separa "enunciado\nA)x\nB)y..." em (enunciado, [{"label","text"}, ...]).

    Ancorado em início de linha (`(?:^|\n)`), não em qualquer ocorrência de "X)" no meio do
    texto — sem isso, alternativas cujo TEXTO contém algo como "(A+C)/B" quebram o parser
    (checado manualmente no CSV: acontece numa das 254 linhas). Também remove um rótulo
    duplicado no início do texto da alternativa ("A)A)3" -> "3"), outro defeito raro da fonte.
    """
    raw_text = str(raw_text)
    parts = _OPT_RE.split(raw_text)
    stem = parts[0].strip()
    labels = parts[1::2]
    texts = []
    for label, text in zip(labels, parts[2::2]):
        text = text.strip()
        text = re.sub(rf"^{re.escape(label)}\)\s*", "", text)
        texts.append(text)
    choices = [{"label": l, "text": t} for l, t in zip(labels, texts)]
    return stem, choices


def option_labels(choices: list[dict]) -> list[str]:
    return [c["label"] for c in choices]


# --- carga + verificação --------------------------------------------------------
df = pd.read_csv(DATA_CSV)
assert (df["rank"] == 1).all(), "esperava rank==1 em todas as linhas (retrieval top-1 já fixado)"
assert df["validation_index"].is_unique, "validation_index deveria ser único (1 vizinho por item)"

n_bad = 0
for col in ("train_question", "validation_question"):
    for raw in df[col]:
        _, choices = parse_choices(raw)
        if option_labels(choices) != ["A", "B", "C", "D", "E"]:
            n_bad += 1
assert n_bad == 0, f"{n_bad} questão(ões) não têm exatamente as alternativas A-E após o parse"

print(f"{len(df)} linhas (pares validação<->treino), rank sempre 1, "
      f"{df['train_index'].nunique()} questões de treino únicas (algumas se repetem como vizinho "
      f"de mais de uma validação).")
print(df[["similarity"]].describe().round(3))
df.head(2)


### Proveniência: por que não usar os splits oficiais por índice

`data/splits/aqua/train.jsonl` tem 383 itens (o subconjunto reduzido que o resto do projeto usa);
`data/splits/aqua/validation.jsonl` tem 252. O `DatasetSpec` do AQuA (`rmcq/config.py`) registra
que o pool bruto tem 97.467 linhas de treino e 254 de validação — os números do CSV (254 pares,
`train_index` na casa das dezenas de milhares) batem com o pool **bruto**, não com o split
reduzido do projeto. Checagem por TEXTO (não por índice) confirma: 0/254 questões de treino do
CSV existem no split oficial de 383; 252/254 questões de validação existem no split oficial (as
outras 2 devem ter sido removidas na deduplicação), e as 252 que batem têm o MESMO gabarito
(`answerKey` oficial == `validation_correct` do CSV, 0 divergências). Por isso os itens deste
notebook são construídos direto do texto do CSV — é a fonte mais completa e autocontida
disponível, e evita qualquer suposição frágil de alinhamento por índice.


In [ ]:
# uid sintético: usa o índice do CSV (estável, único), prefixado para não colidir com uids do
# resto do projeto (aqua-train-000082 etc. são do split OFICIAL reduzido, que isto não é).

def make_item(uid: str, split: str, stem: str, choices: list[dict], gold: str) -> dict:
    assert gold in option_labels(choices), f"{uid}: gabarito {gold!r} não está entre {option_labels(choices)}"
    return {
        "uid": uid,
        "dataset": "aqua",
        "split": split,
        "problem_type": "process",
        "context": None,
        "question": stem,
        "choices": choices,
        "answerKey": gold,
        "num_choices": len(choices),
    }


train_items_by_uid: dict[str, dict] = {}
val_items: list[dict] = []
# pareamento validação -> uid da reflexão de treino a usar, e a similaridade que sustenta isso
val_train_uid: dict[str, str] = {}
val_similarity: dict[str, float] = {}

for _, row in df.iterrows():
    v_uid = f"aqua-ext-validation-{int(row['validation_index'])}"
    t_uid = f"aqua-ext-train-{int(row['train_index'])}"

    if t_uid not in train_items_by_uid:
        stem, choices = parse_choices(row["train_question"])
        train_items_by_uid[t_uid] = make_item(t_uid, "train", stem, choices, row["train_correct"])

    stem, choices = parse_choices(row["validation_question"])
    val_items.append(make_item(v_uid, "validation", stem, choices, row["validation_correct"]))
    val_train_uid[v_uid] = t_uid
    val_similarity[v_uid] = float(row["similarity"])

train_items = list(train_items_by_uid.values())

if SMOKE_TEST:
    val_items = val_items[:SMOKE_LIMIT]
    keep_train = {val_train_uid[it["uid"]] for it in val_items}
    train_items = [it for it in train_items if it["uid"] in keep_train]

print(f"{len(train_items)} questões de treino (únicas) | {len(val_items)} questões de validação")
assert len(val_items) == len(set(it["uid"] for it in val_items))
assert len(train_items) == len(set(it["uid"] for it in train_items))


## Prompts

**Resposta:** a instrução pedida nesta sessão foi literalmente *"in your response, select the
answer by explicitly stating: Answer: X where X is A, B, C or D"*. AQuA tem 5 alternativas
(A-E), não 4 — a lista de letras na instrução é montada dinamicamente a partir das alternativas
de cada item (`A, B, C, D or E` aqui), preservando a frase pedida.

**Reflexão:** reaproveita `rmcq.common.REFLECTION_PROMPTS[(depth, "student")]` — o prompt de
autorreflexão já congelado do projeto (perspectiva "student", porque aqui aluno == professor) —
em vez de reescrever um terceiro do zero. Import direto, não cópia local: não há como ele
divergir do resto do projeto.

**Injeção da reflexão na validação:** formato local e simples (não `build_eval_prompt`/`NOTES_HEADER_V2`
de `rmcq.common`, que estão amarrados à convenção "Lesson:" da v2 e ao formato multi-nota da
grade principal — aqui é sempre 1 nota, então uma instrução direta é mais legível). Duas
mitigações já validadas pelas auditorias do projeto são reaproveitadas mesmo assim:
`neutralize_option_letters` (a reflexão fala de OUTRA questão, "você escolheu D" não pode ancorar
na alternativa D da questão nova) e `compact_reflection` (corta pela cabeça, preserva a lição no
fim). O rótulo "de uma resposta certa/errada" também é preservado, pelo mesmo motivo do resto do
projeto: é o fator mais replicado do histórico (ver reasoning acima).

**Sem reflexão (abaixo do limiar escolhido) o prompt é `build_answer_prompt_v2(item)` puro** —
byte a byte igual ao baseline, por construção (não por uma condição em cascata), exatamente como
`rmcq.common.build_eval_prompt` já garante no resto do projeto.

**Juiz:** recebe a questão, as alternativas, a letra E o texto do gabarito, e a resposta bruta do
modelo — e devolve só `Verdict: CORRECT` ou `Verdict: INCORRECT`. É uma etapa de julgamento
separada da extração por regex (`extract_final_answer`, usada aqui só para diagnóstico de
aderência de formato) porque foi pedido explicitamente: "peça para um modelo de linguagem julgar
se houve acerto".


In [ ]:
def _labels_phrase(choices: list[dict]) -> str:
    labels = option_labels(choices)
    return ", ".join(labels[:-1]) + f" or {labels[-1]}"


def build_answer_prompt_v2(item: dict) -> str:
    return (
        f"You are answering a multiple-choice question.\n\n"
        f"Question: {item['question']}\n\n"
        f"Options:\n{format_options(item['choices'])}\n\n"
        f"Instructions:\n"
        f"- Think step by step before answering.\n"
        f"- Choose exactly one option.\n"
        f"- In your response, select the answer by explicitly stating: "
        f"Answer: X where X is {_labels_phrase(item['choices'])}.\n"
        f"- End your response with that exact line, and nothing after it."
    )


REFLECTION_NOTE_BLOCK = (
    "Below is a note you wrote after answering a DIFFERENT multiple-choice question earlier "
    "(from a {outcome} answer). If it is relevant to the question below, use it to inform your "
    "reasoning; it does not reveal the answer to this question.\n\n"
    "Note:\n{reflection}\n\n"
)


def build_eval_prompt_v2(item: dict, reflection_text: str | None, source_was_correct: bool | None) -> str:
    """Sem reflexão -> build_answer_prompt_v2 puro (mesmo comportamento de rmcq.common.build_eval_prompt)."""
    if not reflection_text:
        return build_answer_prompt_v2(item)
    text = compact_reflection(neutralize_option_letters(reflection_text))
    outcome = "correct" if source_was_correct else "incorrect"
    prefix = REFLECTION_NOTE_BLOCK.format(outcome=outcome, reflection=text)
    return prefix + build_answer_prompt_v2(item)


JUDGE_PROMPT = (
    "You are grading a multiple-choice answer.\n\n"
    "Question: {question}\n\n"
    "Options:\n{options}\n\n"
    "Correct option: {gold_letter}) {gold_text}\n\n"
    "Candidate's response:\n{response}\n\n"
    "Does the candidate's response select the correct option ({gold_letter})? Consider only "
    "which option the candidate ultimately selected, not the quality of their reasoning.\n\n"
    "End your reply with this exact line, and nothing after it:\n"
    "Verdict: <CORRECT or INCORRECT>"
)

_VERDICT_RE = re.compile(r"Verdict\s*:\s*(CORRECT|INCORRECT)", re.IGNORECASE)


def build_judge_prompt(item: dict, response_text: str) -> str:
    gold = item["answerKey"]
    gold_text = next(c["text"] for c in item["choices"] if c["label"] == gold)
    return JUDGE_PROMPT.format(
        question=item["question"],
        options=format_options(item["choices"]),
        gold_letter=gold,
        gold_text=gold_text,
        response=response_text.strip(),
    )


def extract_judge_verdict(text: str) -> bool | None:
    """True/False, ou None se o juiz não seguiu o formato pedido (tratado como abstenção)."""
    hits = _VERDICT_RE.findall(text or "")
    if not hits:
        tail = "\n".join((text or "").splitlines()[-3:]).upper()
        if "INCORRECT" in tail:
            return False
        if "CORRECT" in tail:
            return True
        return None
    return hits[-1].upper() == "CORRECT"


## Backend

Um único modelo carregado quando `STUDENT_MODEL == JUDGE_MODEL` (o caso default): responder,
refletir e julgar são só chamadas com `GenParams` diferentes no mesmo backend. `ANSWER_GEN` é
guloso (`temperature=0`, igual a `STUDENT_GEN` do projeto — é o que torna válido o truque de
reaproveitamento de geração da célula de threshold mais abaixo). `REFLECT_GEN` reaproveita
`TEACHER_GEN` (amostragem a 0.8, para a reflexão não ficar idêntica a cada rerun). `JUDGE_GEN` é
guloso e com poucos tokens — só precisa da linha `Verdict:`.


In [ ]:
student_backend = get_backend(STUDENT_MODEL, kind=BACKEND_KIND_STUDENT)
judge_backend = (
    student_backend if JUDGE_MODEL == STUDENT_MODEL and BACKEND_KIND_JUDGE == BACKEND_KIND_STUDENT
    else get_backend(JUDGE_MODEL, kind=BACKEND_KIND_JUDGE)
)

ANSWER_GEN = GenParams.from_config(STUDENT_GEN, seed=SEED)
REFLECT_GEN = GenParams.from_config(TEACHER_GEN, seed=SEED)
JUDGE_GEN = GenParams.from_config(STUDENT_GEN, seed=SEED, max_new_tokens=64)

print(f"student_backend={student_backend!r}  judge_backend={judge_backend!r}")


## Etapa 1 — `llama3-8b` responde as questões de treino

Prompt baseline puro (`build_answer_prompt_v2`, sem nenhuma reflexão — não pode haver, é o
treino). Gravado em JSONL retomável (`JsonlStore`, mesmo padrão do resto do projeto): interromper
e rodar de novo não regenera o que já foi feito.


In [ ]:
def generate_answers(items: list[dict], store_path: Path, backend, params: GenParams, desc: str) -> JsonlStore:
    store = JsonlStore(store_path)
    done = store.done_keys()
    pending = [it for it in items if it["uid"] not in done]
    if pending:
        prompts = [build_answer_prompt_v2(it) for it in pending]
        gens = backend.generate(prompts, params, desc=desc)
        rows = []
        for item, prompt, gen in zip(pending, prompts, gens):
            ext = extract_final_answer(gen.text, item["choices"])
            rows.append({
                "uid": item["uid"], "split": item["split"], "prompt": prompt,
                "raw_output": gen.text, "regex_predicted": ext.letter,
                "regex_method": ext.method, "regex_followed_format": ext.followed_format,
                "gold": item["answerKey"], "prompt_tokens": gen.prompt_tokens,
                "completion_tokens": gen.completion_tokens,
            })
        store.append(rows)
    log.info("%s: %d/%d já prontos", desc, len(store.done_keys()), len(items))
    return store


train_baseline_store = generate_answers(
    train_items, BASELINE_DIR / STUDENT_MODEL / "train.jsonl", student_backend, ANSWER_GEN,
    desc="treino: resposta baseline",
)
train_baseline_by_uid = {r["uid"]: r for r in train_baseline_store.read_all()}
len(train_baseline_by_uid)


## Etapa 1b — o juiz avalia as respostas de treino

O veredito do juiz vira o feedback (CORRETO/INCORRETO) que a reflexão da etapa 2 recebe, e
também decide quem entra no pool `"errors"` mais adiante. Abstenção do juiz (não seguiu o
formato `Verdict:` nem escreveu CORRECT/INCORRECT em nenhuma forma reconhecível) é tratada como
incorreta para fins de métrica, igual à convenção do resto do projeto para abstenção do aluno
(`rmcq.stages.analyze.utility`) — mas fica marcada em `judge_abstained` para auditoria, nunca
escondida.


In [ ]:
def judge_answers(items_by_uid: dict[str, dict], answers_by_uid: dict[str, dict],
                   store_path: Path, desc: str) -> JsonlStore:
    store = JsonlStore(store_path)
    done = store.done_keys()
    pending_uids = [u for u in answers_by_uid if u not in done]
    if pending_uids:
        prompts = [build_judge_prompt(items_by_uid[u], answers_by_uid[u]["raw_output"]) for u in pending_uids]
        gens = judge_backend.generate(prompts, JUDGE_GEN, desc=desc)
        rows = []
        for uid, prompt, gen in zip(pending_uids, prompts, gens):
            verdict = extract_judge_verdict(gen.text)
            rows.append({
                "uid": uid, "judge_prompt": prompt, "judge_raw_output": gen.text,
                "is_correct": bool(verdict) if verdict is not None else False,
                "judge_abstained": verdict is None,
            })
        store.append(rows)
    log.info("%s: %d/%d já prontos", desc, len(store.done_keys()), len(answers_by_uid))
    return store


train_items_by_uid = {it["uid"]: it for it in train_items}
train_judge_store = judge_answers(
    train_items_by_uid, train_baseline_by_uid, JUDGE_DIR / JUDGE_MODEL / "train_baseline.jsonl",
    desc="treino: juiz",
)
train_correct_by_uid = {r["uid"]: r["is_correct"] for r in train_judge_store.read_all()}

n_correct = sum(train_correct_by_uid.values())
n_abstained = sum(1 for r in train_judge_store.read_all() if r["judge_abstained"])
print(f"treino: {n_correct}/{len(train_correct_by_uid)} corretas pelo juiz "
      f"({n_abstained} abstenções do juiz)")


## Etapa 2 — autorreflexão sobre cada resposta de treino, por profundidade

Usa `REFLECTION_PROMPTS[(depth, "student")]` de `rmcq.common` (importado, não copiado — ver
seção de prompts) com o veredito do juiz como feedback. Gerado para as duas profundidades
(`simple`, `complex`) — barato (252 itens × 2), serve de checagem de robustez (ver reasoning).


In [ ]:
def build_reflection_prompt_local(item: dict, previous_answer: str, was_correct: bool, depth: str) -> str:
    instruction = REFLECTION_PROMPTS[(depth, "student")]
    feedback = FEEDBACK_CORRECT if was_correct else FEEDBACK_INCORRECT
    return (
        f"{instruction}\n\n"
        f"Question: {item['question']}\n\n"
        f"Options:\n{format_options(item['choices'])}\n\n"
        f"Your previous answer:\n{previous_answer.strip()}\n\n"
        f"{feedback}"
    )


reflections_by_depth: dict[str, dict[str, dict]] = {}

for depth in DEPTHS:
    store = JsonlStore(REFLECTIONS_DIR / depth / "train.jsonl")
    done = store.done_keys()
    pending = [it for it in train_items if it["uid"] not in done]
    if pending:
        prompts = [
            build_reflection_prompt_local(
                it, train_baseline_by_uid[it["uid"]]["raw_output"],
                train_correct_by_uid[it["uid"]], depth,
            )
            for it in pending
        ]
        gens = student_backend.generate(prompts, REFLECT_GEN, desc=f"reflexao treino ({depth})")
        rows = [
            {
                "uid": it["uid"], "depth": depth, "prompt": prompt,
                "reflection_text": strip_think(gen.text),
                "source_was_correct": train_correct_by_uid[it["uid"]],
            }
            for it, prompt, gen in zip(pending, prompts, gens)
        ]
        store.append(rows)
    reflections_by_depth[depth] = {r["uid"]: r for r in store.read_all()}
    log.info("reflexões (%s): %d/%d", depth, len(reflections_by_depth[depth]), len(train_items))

{d: len(v) for d, v in reflections_by_depth.items()}


## Etapa 3 — baseline de validação (o controle)

Mesmas 254 questões de validação, prompt baseline puro. É o denominador de toda comparação
daqui pra frente (`utility()` é sempre pareado contra isto).


In [ ]:
val_baseline_store = generate_answers(
    val_items, BASELINE_DIR / STUDENT_MODEL / "validation.jsonl", student_backend, ANSWER_GEN,
    desc="validação: resposta baseline",
)
val_baseline_by_uid = {r["uid"]: r for r in val_baseline_store.read_all()}

val_items_by_uid = {it["uid"]: it for it in val_items}
val_baseline_judge_store = judge_answers(
    val_items_by_uid, val_baseline_by_uid, JUDGE_DIR / JUDGE_MODEL / "validation_baseline.jsonl",
    desc="validação: juiz (baseline)",
)
val_baseline_correct = {r["uid"]: r["is_correct"] for r in val_baseline_judge_store.read_all()}

acc_baseline = sum(val_baseline_correct.values()) / len(val_baseline_correct)
print(f"acurácia baseline (validação, sem reflexão): {acc_baseline:.3f} "
      f"({sum(val_baseline_correct.values())}/{len(val_baseline_correct)})")


## Etapa 4 — validação com a reflexão do par de treino, incondicional

Gerado **uma vez por profundidade, para todos os 254 itens**, injetando sempre a reflexão do
vizinho mais similar (sem aplicar limiar ainda). Como `ANSWER_GEN` é guloso, isso é
matematicamente equivalente a gerar só para quem passa em cada limiar — a etapa 5 reaproveita
este resultado para todo o grid de `threshold × pool` sem chamar o modelo de novo (ver
reasoning). Se algum item de validação não tiver reflexão disponível (não deveria acontecer aqui,
o CSV garante rank=1 para as 254 linhas, mas a checagem fica pelo caso geral), o prompt cai no
baseline puro.


In [ ]:
val_with_reflection_by_uid: dict[str, dict[str, dict]] = {}
val_with_reflection_correct: dict[str, dict[str, bool]] = {}

for depth in DEPTHS:
    refl = reflections_by_depth[depth]
    store = JsonlStore(EVAL_DIR / depth / "validation.jsonl")
    done = store.done_keys()
    pending = [it for it in val_items if it["uid"] not in done]
    if pending:
        prompts = []
        for it in pending:
            t_uid = val_train_uid[it["uid"]]
            r = refl.get(t_uid)
            reflection_text = r["reflection_text"] if r else None
            source_correct = r["source_was_correct"] if r else None
            prompts.append(build_eval_prompt_v2(it, reflection_text, source_correct))
        gens = student_backend.generate(prompts, ANSWER_GEN, desc=f"validação com reflexão ({depth})")
        rows = []
        for it, prompt, gen in zip(pending, prompts, gens):
            ext = extract_final_answer(gen.text, it["choices"])
            rows.append({
                "uid": it["uid"], "depth": depth, "prompt": prompt, "raw_output": gen.text,
                "regex_predicted": ext.letter, "regex_followed_format": ext.followed_format,
                "train_uid": val_train_uid[it["uid"]], "similarity": val_similarity[it["uid"]],
                "had_reflection": val_train_uid[it["uid"]] in refl,
            })
        store.append(rows)
    by_uid = {r["uid"]: r for r in store.read_all()}
    val_with_reflection_by_uid[depth] = by_uid

    judge_store = judge_answers(
        val_items_by_uid, by_uid, JUDGE_DIR / JUDGE_MODEL / f"validation_with_reflection__{depth}.jsonl",
        desc=f"validação com reflexão: juiz ({depth})",
    )
    val_with_reflection_correct[depth] = {r["uid"]: r["is_correct"] for r in judge_store.read_all()}
    acc = sum(val_with_reflection_correct[depth].values()) / len(val_with_reflection_correct[depth])
    print(f"[{depth}] acurácia com reflexão incondicional: {acc:.3f}")


## Etapa 5 — montar o grid (threshold × pool), sem gerar nada novo

Por item de validação, decide se USA a resposta "com reflexão" (etapa 4) ou cai na resposta
baseline (etapa 3) — a definição operacional de "nenhuma reflexão recuperada" pedida pelo
usuário. Dois filtros, aplicados em AND:

- **threshold**: `similarity(item) >= threshold` (a similaridade já vem do CSV).
- **pool**: `"all"` sempre libera; `"errors"` só libera quando o par de treino foi **errado**
  pelo modelo (`train_correct_by_uid[train_uid] is False`).


In [ ]:
def assemble_condition(depth: str, threshold: float, pool: str) -> dict[str, dict]:
    rows = {}
    for it in val_items:
        uid = it["uid"]
        t_uid = val_train_uid[uid]
        sim = val_similarity[uid]
        eligible = sim >= threshold
        if pool == "errors":
            eligible = eligible and (train_correct_by_uid.get(t_uid) is False)
        use_reflection = eligible and val_with_reflection_by_uid[depth][uid]["had_reflection"]

        if use_reflection:
            answer_row = val_with_reflection_by_uid[depth][uid]
            is_correct = val_with_reflection_correct[depth][uid]
        else:
            answer_row = val_baseline_by_uid[uid]
            is_correct = val_baseline_correct[uid]

        rows[uid] = {
            "uid": uid, "is_correct": is_correct, "used_reflection": use_reflection,
            "similarity": sim, "extra": {"top1_similarity": sim},
        }
    return rows


grid_conditions: dict[tuple[str, float, str], dict[str, dict]] = {}
for depth in DEPTHS:
    for threshold in THRESHOLDS:
        for pool in POOLS:
            rows = assemble_condition(depth, threshold, pool)
            grid_conditions[(depth, threshold, pool)] = rows
            tag = f"{depth}__t{threshold:.2f}__pool{pool}"
            JsonlStore(GRID_DIR / tag / "validation.jsonl").append(list(rows.values()))

len(grid_conditions), next(iter(grid_conditions))


## Etapa 6 — métricas: utility, McNemar exato, cobertura

`utility()` é importado de `rmcq.stages.analyze` — mesma definição do resto do projeto
((errado→certo) − (certo→errado), pareado contra o baseline). O teste exato de McNemar é
implementado com `math.comb` (binomial exata sobre os pares discordantes, sem depender de
`scipy`, que não está disponível neste ambiente) — equivalente a
`scipy.stats.binomtest(k, n, 0.5)` bicaudal.


In [ ]:
baseline_rows = {
    uid: {"uid": uid, "is_correct": val_baseline_correct[uid], "extra": {}}
    for uid in val_baseline_correct
}


def mcnemar_exact_p(b: int, c: int) -> float:
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    cum = sum(math.comb(n, i) for i in range(0, k + 1)) / (2 ** n)
    return min(1.0, 2 * cum)


summary_rows = []
for (depth, threshold, pool), rows in grid_conditions.items():
    block = utility(baseline_rows, rows)
    p = mcnemar_exact_p(block["wrong_to_right"], block["right_to_wrong"])
    coverage = sum(1 for r in rows.values() if r["used_reflection"]) / len(rows)
    summary_rows.append({
        "depth": depth, "threshold": threshold, "pool": pool,
        "n": block["n_shared"], "coverage_reflection_used": round(coverage, 3),
        "baseline_accuracy": block["baseline_accuracy"],
        "condition_accuracy": block["condition_accuracy"],
        "utility": block["utility"], "wrong_to_right": block["wrong_to_right"],
        "right_to_wrong": block["right_to_wrong"], "mcnemar_p": round(p, 4),
        "is_anchor": threshold == ANCHOR_THRESHOLD and pool == ANCHOR_POOL,
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["depth", "threshold", "pool"]).reset_index(drop=True)
summary_df.to_csv(DIAG_DIR / "summary_grid.csv", index=False)
summary_df


## Condição-âncora vs. o resto do grid

A comparação que decide alguma coisa é `threshold=0.0, pool="all"` (usa a reflexão do vizinho
mais similar em toda a validação) contra o baseline, por profundidade — definida ANTES de rodar
(ver reasoning). As outras 14 células do grid abaixo são sensibilidade, não uma segunda chance de
achar significância: com 16 testes, ~1 p<0.05 é esperado só por acaso (α=0.05 × 16 ≈ 0.8).


In [ ]:
anchor = summary_df[summary_df["is_anchor"]].copy()
print("=== condição-âncora (threshold=0.0, pool=all) — a comparação primária ===")
for _, r in anchor.iterrows():
    sig = "p<0.05" if r["mcnemar_p"] < 0.05 else "n.s."
    print(f"[{r['depth']:>7}] baseline={r['baseline_accuracy']:.3f}  com_reflexao={r['condition_accuracy']:.3f}  "
          f"utility={r['utility']:+.4f}  (errado→certo={r['wrong_to_right']}, certo→errado={r['right_to_wrong']}, "
          f"McNemar p={r['mcnemar_p']:.4f}, {sig})")

print()
print("=== resto do grid (sensibilidade — não escolher a célula 'vencedora' depois de ver os números) ===")
rest = summary_df[~summary_df["is_anchor"]].sort_values("utility")
rest


## Plots

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    for depth in DEPTHS:
        sub = summary_df[summary_df["depth"] == depth].sort_values("threshold")
        sub_all = sub[sub["pool"] == "all"]
        sub_err = sub[sub["pool"] == "errors"]
        axes[0].plot(sub_all["threshold"], sub_all["condition_accuracy"], marker="o", label=f"{depth} / all")
        axes[0].plot(sub_err["threshold"], sub_err["condition_accuracy"], marker="s", linestyle="--", label=f"{depth} / errors")
    axes[0].axhline(acc_baseline, color="black", linestyle=":", label="baseline")
    axes[0].set_xlabel("limiar de similaridade")
    axes[0].set_ylabel("acurácia (validação)")
    axes[0].set_title("Acurácia vs. limiar")
    axes[0].legend(fontsize=7)

    pivot = summary_df.pivot_table(index=["depth", "pool"], columns="threshold", values="utility")
    im = axes[1].imshow(pivot.values, cmap="RdBu", vmin=-0.15, vmax=0.15, aspect="auto")
    axes[1].set_xticks(range(len(pivot.columns)))
    axes[1].set_xticklabels([f"{t:.2f}" for t in pivot.columns])
    axes[1].set_yticks(range(len(pivot.index)))
    axes[1].set_yticklabels([f"{d}/{p}" for d, p in pivot.index])
    axes[1].set_title("Utility (errado→certo − certo→errado) / n")
    fig.colorbar(im, ax=axes[1], fraction=0.046)

    fig.tight_layout()
    fig.savefig(DIAG_DIR / "grid_overview.png", dpi=150)
    plt.show()
except ImportError:
    print("matplotlib não disponível neste ambiente — pulei os gráficos. "
          f"Os números completos estão em {DIAG_DIR / 'summary_grid.csv'}.")


## Próximos passos

- `SMOKE_TEST = True` acima usa o `StubBackend` (sem GPU) e limita a validação a
  `SMOKE_LIMIT` itens — é só para validar a canalização. Trocar para `False` (e checar
  `RMCQ_BACKEND` / `RMCQ_ACTIVE_MODELS` no `.env`) para a rodada de verdade nos 254 itens.
- **No modo smoke, o juiz abstém de 100% dos itens (acurácia sai 0/0 em tudo) — é esperado, não
  um bug.** O `StubBackend` do projeto (`rmcq/backends/stub.py`) sabe simular `FINAL ANSWER: X`
  (o formato do resto do projeto), mas não conhece o formato `Verdict: CORRECT/INCORRECT` pedido
  aqui — então `extract_judge_verdict` corretamente não acha nada e conta como abstenção. A
  canalização (extração, filtros de threshold/pool, `utility()`, McNemar) roda e não quebra com
  tudo zerado, que é o que este smoke test verifica; com um backend de verdade (`vllm`/`hf`/
  `azure`) o juiz segue a instrução normalmente.
- Se o ambiente tiver o gateway Azure da Petrobras disponível, considere trocar `JUDGE_MODEL`
  para um deployment registrado (ver nota na célula de config) — reduz o risco de o modelo ser
  complacente ao julgar a própria resposta.
- Nenhuma saída de execução real foi entregue com este notebook (mesmo padrão dos notebooks
  03/04/05) — rodar com `SMOKE_TEST = False` e interpretar os números é trabalho do usuário.
